In [2]:
import pickle
import pandas as pd

with open("../cached_training_data.pkl", "rb") as f:
    data = pickle.load(f)

print(type(data))
if isinstance(data, dict):
    print(data.keys())


<class 'pandas.core.frame.DataFrame'>


In [3]:
# Convert to DataFrame if not already one
if isinstance(data, pd.DataFrame):
    df = data
elif isinstance(data, dict):
    df = pd.DataFrame(data)
else:
    df = pd.DataFrame(data)

print(f"Shape: {df.shape}")
df.head()


Shape: (34321, 245)


,prod_act_sum,pinst_active_sum_lag_0_hours,log_wind_fc_lag_0_hours,log_wind_fc_lag_1_hours,log_wind_fc_lag_2_hours,log_wind_fc_lag_24_hours,log_wind_fc_lag_168_hours,log_wind_fc_w_hour_of_day_7_mean,log_wind_fc_w_hour_of_day_7_std,log_wind_fc_w_hour_of_day_7_slope,...,afrr_pos_avg_price_act_w_hour_of_day_7_slope,afrr_pos_avg_price_act_w_hour_of_day_14_mean,afrr_pos_avg_price_act_w_hour_of_day_14_std,afrr_pos_avg_price_act_w_hour_of_day_14_slope,afrr_pos_avg_price_act_w_hour_of_day_28_mean,afrr_pos_avg_price_act_w_hour_of_day_28_std,afrr_pos_avg_price_act_w_hour_of_day_28_slope,afrr_pos_avg_price_act_w_past_hours_42_48_mean,afrr_pos_avg_price_act_w_past_hours_42_48_std,afrr_pos_avg_price_act_w_past_hours_42_48_slope
2022-02-01 00:00:00+01:00,754.901425,1188.466729,5.396350,NaN,NaN,NaN,NaN,5.396350,NaN,NaN,...,NaN,15.427042,NaN,NaN,15.427042,NaN,NaN,NaN,NaN,NaN
2022-02-01 01:00:00+01:00,776.544487,1208.662577,5.459622,5.396350,NaN,NaN,NaN,5.427986,0.044740,0.031636,...,0.321774,15.748817,0.455057,0.321774,15.748817,0.455057,0.321774,NaN,NaN,NaN
2022-02-01 02:00:00+01:00,838.167161,1210.035942,5.484049,5.459622,5.396350,NaN,NaN,5.446674,0.045260,0.029233,...,0.160573,15.802132,0.334763,0.160573,15.802132,0.334763,0.160573,NaN,NaN,NaN
2022-02-01 03:00:00+01:00,897.355137,1210.378591,5.437109,5.484049,5.459622,NaN,NaN,5.444282,0.037263,0.010190,...,0.035696,15.744055,0.296989,0.035696,15.744055,0.296989,0.035696,NaN,NaN,NaN
2022-02-01 04:00:00+01:00,870.331323,1181.189603,5.433557,5.437109,5.484049,NaN,NaN,5.442137,0.032625,0.007441,...,-0.042813,15.637840,0.350086,-0.042813,15.637840,0.350086,-0.042813,NaN,NaN,NaN


In [4]:
# Count NaN values per column
nan_counts = df.isna().sum().sort_values(ascending=False)
nan_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)

nan_summary = pd.DataFrame({
    "nan_count": nan_counts,
    "nan_pct": nan_pct.round(2),
    "total_rows": len(df)
})

print(f"Total rows: {len(df)}")
print(f"Columns with NaNs: {(nan_counts > 0).sum()} / {len(df.columns)}")
print()
print(nan_summary.to_string())


Total rows: 34321
Columns with NaNs: 195 / 245

                                                 nan_count  nan_pct  total_rows
prod_act_sum_lag_168_hours                             168     0.49       34321
fcr_price_act_lag_168_hours                            168     0.49       34321
solar_fc_lag_168_hours                                 168     0.49       34321
afrr_pos_avg_price_act_lag_168_hours                   168     0.49       34321
log_wind_fc_lag_168_hours                              168     0.49       34321
missing_pinst_sum_lag_168_hours                        168     0.49       34321
spot_price_act_lag_168_hours                           168     0.49       34321
rebap_price_est_lag_168_hours                          168     0.49       34321
afrr_neg_avg_price_act_lag_168_hours                   168     0.49       34321
prod_fc_sum_lag_168_hours                              168     0.49       34321
prod_act_sum_lag_96_hours                               96     0.28     

In [5]:
import matplotlib.pyplot as plt

col = "cov_prod_act_sum_lag_96_hours"
if col in df.columns:
    is_nan = df[col].isna().astype(int)

    # Rolling NaN rate (7-day window)
    window = 24 * 7
    nan_rolling = is_nan.rolling(window, min_periods=1).mean() * 100

    fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

    # 1) Raw values with NaN markers
    axes[0].plot(df.index, df[col], linewidth=0.5, label="value")
    nan_idx = df.index[df[col].isna()]
    axes[0].scatter(nan_idx, [df[col].min()] * len(nan_idx), c="red", s=1, label="NaN")
    axes[0].set_ylabel("Value")
    axes[0].set_title(f"{col} — values (red = NaN positions)")
    axes[0].legend()

    # 2) Binary NaN indicator
    axes[1].fill_between(df.index, is_nan, step="mid", alpha=0.5, color="red")
    axes[1].set_ylabel("Is NaN")
    axes[1].set_title("NaN indicator (1 = missing)")

    # 3) Rolling NaN percentage
    axes[2].plot(df.index, nan_rolling, color="darkred")
    axes[2].set_ylabel("NaN %")
    axes[2].set_title(f"Rolling NaN rate ({window}h window)")
    axes[2].set_xlabel("Date")

    plt.tight_layout()
    plt.show()

    # Monthly NaN summary
    monthly = is_nan.groupby(df.index.to_period("M")).agg(["sum", "count"])
    monthly.columns = ["nan_count", "total"]
    monthly["nan_pct"] = (monthly["nan_count"] / monthly["total"] * 100).round(1)
    print(f"\nMonthly NaN summary for {col}:")
    print(monthly.to_string())
else:
    print(f"Column '{col}' not found in DataFrame")


Column 'cov_prod_act_sum_lag_96_hours' not found in DataFrame
